In [1]:
pip install langchain pydantic chromadb tiktoken sentence-transformers langgraph


  Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_semantic_conventions-0.66b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl (140 kB)
Using cached opentelemetry_api-1.45.1-py3-none-any.whl (60 kB)
Using cached opentelemetry_semantic_conventions-0.66b1-py3-none-any.whl (206 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelem

In [2]:
pip install "opentelemetry-api<=1.42.1" "opentelemetry-sdk<=1.42.1"

  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.45.1
    Uninstalling opentelemetry-api-1.45.1:
      Successfully uninstalled opentelemetry-api-1.45.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.66b1
    Uninstalling opentelemetry-semantic-conventions-0.66b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.66b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelem

In [3]:
!pip install langchain-community

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Pydantic documentation ka URL
url = "https://docs.pydantic.dev/"

# 2. Document load karna
loader = WebBaseLoader(url)
docs = loader.load()

# 3. Chunks mein split karna
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(chunks)}")

/tmp/ipykernel_20293/637288739.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Total chunks created: 4


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Open-source embedding model initialize karna (HuggingFace se)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Vector store (ChromaDB) mein chunks aur embeddings store karna
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pydantic_docs"
)

# 3. Retriever banana taaki baad mein query ke liye use kar sakein
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Vector store successfully created and indexed!")

/tmp/ipykernel_20293/1715730780.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store successfully created and indexed!


In [6]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage

# Graph State define karna
class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]
    retry_count: int

print("GraphState schema defined successfully!")

GraphState schema defined successfully!


In [7]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LoVlCGFgTqJxC5r1xkDh88WuWO_q0XUHh52n3r9NYybQ"

In [8]:
!pip install langchain-google-genai

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

# LLM setup (Aap Gemini ya koi bhi LLM use kar sakte hain)
# For example, using ChatGoogleGenerativeAI:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# Node 1: Query Analysis (Rewrite/Expand query)
def query_analysis(state: GraphState):
    print("---NODE 1: QUERY ANALYSIS---")
    question = state["question"]

    # Simple query rewriter prompt
    prompt = ChatPromptTemplate.from_template(
        "You are an expert at refining technical search queries. Rewrite the following user question to make it more precise for vector search: \n\n Question: {question}"
    )
    chain = prompt | llm | StrOutputParser()
    better_query = chain.invoke({"question": question})

    return {"question": better_query}

# Node 2: Retrieval
def retrieve_documents(state: GraphState):
    print("---NODE 2: RETRIEVAL---")
    question = state["question"]
    # Humne jo pehle retriever banaya tha uska use karenge
    docs = retriever.invoke(question)
    document_texts = [doc.page_content for doc in docs]
    return {"documents": document_texts}

print("Query Analysis and Retrieval nodes defined successfully!")

Query Analysis and Retrieval nodes defined successfully!


In [9]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM सेटअप (Gemini Flash)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# Node 3: Document Grading (दस्तावेज़ों का मूल्यांकन)
def grade_documents(state):
  print("---CHECK DOCUMENT RELEVANCE TO QUERY---")
  question = state["question"]
  documents = state["documents"]

  # प्रॉम्ट टेम्प्लेट (दस्तावेज़ प्रासंगिक है या नहीं यह जांचने के लिए)
  prompt = ChatPromptTemplate.from_messages([
      (
          "system",
          "आप एक ग्रिडर हैं जो उपयोगकर्ता के प्रश्न के खिलाफ पुनर्प्राप्त दस्तावेज़ की प्रासंगिकता का मूल्यांकन कर रहा है।"
          " यदि दस्तावेज़ में प्रश्न से संबंधित कोई कीवर्ड या अर्थ है, तो इसे प्रासंगिक के रूप में रेट करें।"
          " JSON प्रारूप में केवल एक कुंजी 'score' दें जिसमें 'yes' या 'no' हो।",
      ),
      (
          "human",
          "प्रश्न: {question} \n\n दस्तावेज़: {context} \n\n क्या यह प्रासंगिक है? (yes/no)",
      ),
  ])

  scoring_chain = prompt | llm | JsonOutputParser()

  filtered_docs = []
  web_search_needed = False

  for d in documents:
    score = scoring_chain.invoke({"question": question, "context": d.page_content})
    if score.get("score") == "yes":
      filtered_docs.append(d)
    else:
      continue

  # यदि कोई दस्तावेज़ प्रासंगिक नहीं मिला
  if not filtered_docs:
    web_search_needed = True

  return {
      "documents": filtered_docs,
      "question": question,
      "web_search": web_search_needed,
  }


# Node 4: Generation (अंतिम उत्तर जनरेशन)
def generate_answer(state):
  print("---GENERATE ANSWER---")
  question = state["question"]
  documents = state["documents"]

  # उत्तर तैयार करने के लिए प्रॉम्ट
  prompt = ChatPromptTemplate.from_messages([
      (
          "system",
          "आप तकनीकी दस्तावेज़ों के आधार पर सटीक उत्तर देने वाले सहायक हैं।"
          " दिए गए संदर्भ (context) का उपयोग करके स्पष्ट और सटीक उत्तर दें।"
          " यदि संदर्भ अपर्याप्त है, तो बताएं कि आपको उत्तर नहीं पता है।"
      ),
      ("human", "संदर्भ:\n{context}\n\nप्रश्न: {question}"),
  ])

  context_str = "\n\n".join([d.page_content for d in documents])
  generation_chain = prompt | llm

  response = generation_chain.invoke(
      {"context": context_str, "question": question}
  )

  return {"documents": documents, "question": question, "generation": response}

In [10]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

sample_docs = [
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain."
    ),
    Document(
        page_content="FastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on standard Python type hints."
    ),
]

# Text Splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_list = text_splitter.split_documents(sample_docs)

# Vector Store Setup using the working HuggingFace Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=docs_list, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Node 1: Query Analysis (प्रश्न का विश्लेषण और सुधार)
def query_analysis(state):
    print("---ANALYZE & REWRITE QUERY---")
    question = state["question"]
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "आप एक सहायक हैं जो उपयोगकर्ता के प्रश्न को बेहतर खोज परिणामों के लिए दोबारा लिखते हैं।"
            " मूल प्रश्न को अधिक स्पष्ट और कीवर्ड-समृद्ध बनाएं। केवल सुधरा हुआ प्रश्न आउटपुट दें。",
        ),
        ("human", "मूल प्रश्न: {question}"),
    ])
    rewriter = prompt | llm
    better_question = rewriter.invoke({"question": question})

    # Safely extract text content regardless of format
    if hasattr(better_question, "content"):
        better_question = better_question.content

    if isinstance(better_question, list):
        better_question = "".join([str(item) for item in better_question])

    better_question = str(better_question).strip()

    return {"question": better_question, "documents": state.get("documents", [])}

# Node 2: Retrieval (दस्तावेज़ खोजना)
def retrieve(state):
  print("---RETRIEVE DOCUMENTS---")
  question = state["question"]
  documents = retriever.invoke(question)
  return {"documents": documents, "question": question}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
from langgraph.graph import END, StateGraph, START
from typing import List, Any
from typing_extensions import TypedDict

# 1. राउटिंग डिसाइड करने का फंक्शन (Conditional Edge के लिए)
def decision_to_generate(state):
    print("---ASSESS GRADED DOCUMENTS---")
    if state["web_search"]:
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUERY, RE-WRITE QUERY---")
        return "transform_query"
    else:
        print("---DECISION: GENERATE---")
        return "generate"

# 2. ग्राफ स्टेट स्कीमा (Updated to accept any object/Document types)
class GraphState(TypedDict):
    question: str
    generation: Any
    web_search: bool
    documents: List[Any]

workflow = StateGraph(GraphState)

# 3. नोट्स (Nodes) को जोड़ें
workflow.add_node("query_analysis", query_analysis)  # Node 1
workflow.add_node("retrieve", retrieve)  # Node 2
workflow.add_node("grade_documents", grade_documents)  # Node 3
workflow.add_node("generate", generate_answer)  # Node 4

# 4. एजेज (Edges) और कंडीशनल लॉजिक सेट करें
workflow.add_edge(START, "query_analysis")
workflow.add_edge("query_analysis", "retrieve")
workflow.add_edge("retrieve", "grade_documents")

# कंडीशनल एज जोड़ें
workflow.add_conditional_edges(
    "grade_documents",
    decision_to_generate,
    {"transform_query": "query_analysis", "generate": "generate"},
)

workflow.add_edge("generate", END)

# 5. ग्राफ को कंपाइल करें
graph_app = workflow.compile()
print("---GRAPH COMPILED SUCCESSFULLY---")

---GRAPH COMPILED SUCCESSFULLY---


In [12]:
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio

# नेस्टेड एसिंक लूप को एनेबल करें
nest_asyncio.apply()

app = FastAPI(title="RAG Technical Documentation Assistant")

class QueryRequest(BaseModel):
    question: str

class QueryResponse(BaseModel):
    question: str
    generation: str
    documents_count: int

@app.post("/query", response_model=QueryResponse)
async def query_endpoint(request: QueryRequest):
    try:
        inputs = {"question": request.question}
        global_vars = globals()
        compiled_graph = global_vars.get("graph_app")
        if compiled_graph is None:
            raise HTTPException(status_code=500, detail="LangGraph 'graph_app' is not compiled.")

        result = compiled_graph.invoke(inputs)

        raw_gen = result.get("generation", "")

        # सुरक्षित तरीके से स्ट्रिंग एक्सट्रैक्ट करें
        if hasattr(raw_gen, "content"):
            generation_content = str(raw_gen.content)
        elif isinstance(raw_gen, list):
            generation_content = "".join([str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in raw_gen])
        elif isinstance(raw_gen, dict):
            generation_content = str(raw_gen.get("text", str(raw_gen)))
        else:
            generation_content = str(raw_gen)

        docs_count = len(result.get("documents", []))

        return {
            "question": request.question,
            "generation": generation_content.strip(),
            "documents_count": docs_count,
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/documents")
async def list_documents():
    return {
        "indexed_documents": [
            doc.page_content[:100] + "..." for doc in docs_list
        ]
    }

print("--- FASTAPI APP CONFIGURED ---")

--- FASTAPI APP CONFIGURED ---


In [13]:
import uvicorn
import threading
import asyncio

def run_server():
    config = uvicorn.Config(app=app, host="127.0.0.1", port=8006, log_level="info", loop="asyncio")
    server = uvicorn.Server(config)
    asyncio.run(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started successfully in background on port 8006!")

FastAPI server started successfully in background on port 8006!


In [14]:
import requests

try:
    response = requests.get("http://127.0.0.1:8006/documents")
    print("GET /documents Response:", response.json())
except Exception as e:
    print("GET /documents failed:", e)

try:
    query_payload = {"question": "What is LangGraph?"}
    query_response = requests.post("http://127.0.0.1:8006/query", json=query_payload)
    print("\nPOST /query Response:")
    print(query_response.json())
except Exception as e:
    print("\nPOST /query failed:", e)

INFO:     127.0.0.1:47246 - "GET /documents HTTP/1.1" 200 OK
GET /documents Response: {'indexed_documents': ['LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of La...', 'FastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on s...']}
---ANALYZE & REWRITE QUERY---
---RETRIEVE DOCUMENTS---
---CHECK DOCUMENT RELEVANCE TO QUERY---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
---GENERATE ANSWER---
INFO:     127.0.0.1:47250 - "POST /query HTTP/1.1" 200 OK

POST /query Response:
{'question': 'What is LangGraph?', 'generation': "[{'type': 'text', 'text': 'दिए गए संदर्भ के अनुसार:\\n\\n* **LangGraph क्या है:** यह LLMs (लार्ज लैंग्वेज मॉडल्स) के साथ स्टेटफुल (stateful) और मल्टी-एक्टर (multi-actor) ऍप्लिकेशन्स बनाने के लिए एक लाइब्रेरी है, जिसे LangChain के ऊपर बनाया गया है।\\n* **उपयोग (Use Cases):** इसका मुख्य उपयोग LLMs का उपयोग करके ऐसी ऍप्लिकेशन्स का निर्माण करना है जिनमें स्थिति (state) बनाए 

In [13]:
!pip install -U langchain langchain-core langsmith langgraph langchain-google-genai

In [ ]:
!pip install -U langchain langchain-core langsmith langgraph